In [4]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
from openai import OpenAI

In [5]:
gemini_api_key=os.getenv('GEMINI_API_KEY')
groq_api_key=os.getenv('GROQ_API_KEY')

In [6]:
openAI = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)
gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [7]:
import gradio as gr

In [9]:
system_message="You are a helpful assistant"
def message_gpt(prompt):
    messages=[{"role":"system","content":system_message},{"role":"user","content":prompt}]
    response=openAI.chat.completions.create(model="openai/gpt-oss-120b",messages=messages)
    return response.choices[0].message.content

In [10]:
message_gpt("tell me a joke")

'Why don’t scientists trust atoms?\n\nBecause they make up everything!'

In [12]:
message_input=gr.Textbox(label="Your message:",info="Enter the message for gpt-oss",lines=7)
message_output=gr.Textbox(label="Response",lines=8)
view=gr.Interface(fn=message_gpt,title="gpt-oss",inputs=[message_input],outputs=[message_output],examples=["hello","hi there"],flagging_mode="never")
view.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [13]:
system_message="You are a helpful assistant that responds in markdown without code blocks"
message_input=gr.Textbox(label="Your message:",info="Enter the message for gpt-oss",lines=7)
message_output=gr.Textbox(label="Markdown")
view=gr.Interface(fn=message_gpt,title="gpt-oss",inputs=[message_input],outputs=[message_output],examples=["explain transformer architecture to a child"],flagging_mode="never")
view.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [14]:
def stream_gpt(prompt):
    messages=[{"role":"system","content":system_message},{"role":"user","content":prompt}]
    stream=openAI.chat.completions.create(model="openai/gpt-oss-120b",messages=messages,stream=True)
    result=""
    for chunk in stream:
        result+=chunk.choices[0].delta.content or ""
        yield result

In [18]:
system_message="You are a helpful assistant that responds in markdown without code blocks"
message_input=gr.Textbox(label="Your message:",info="Enter the message for gpt-oss",lines=7)
message_output=gr.Markdown(label="Markdown")
view=gr.Interface(fn=stream_gpt,title="gpt-oss",inputs=[message_input],outputs=[message_output],examples=["explain transformer architecture to a child"],flagging_mode="never")
view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [16]:
def stream_gemini(prompt):
    messages=[{"role":"system","content":system_message},{"role":"user","content":prompt}]
    stream=gemini.chat.completions.create(model="gemini-2.5-flash",messages=messages,stream=True)
    result=""
    for chunk in stream:
        result+=chunk.choices[0].delta.content or ""
        yield result

In [17]:
def stream_model(prompt,model):
    if model=="GPT":
        result=stream_gpt(prompt)
    elif model=="Gemini":
        result=stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [22]:
message_input=gr.Textbox(label="Your message:",info="Enter the message for the LLM",lines=7)
model_selector=gr.Dropdown(["GPT","Gemini"],label="Select model",value="GPT")
message_output=gr.Markdown(label="Response")
view=gr.Interface(fn=stream_model,title="LLMs",inputs=[message_input,model_selector],outputs=[message_output],examples=[["explain transformer architecture to a child"],["explain attention mechanism in transformer"]],flagging_mode="never")
view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [23]:
brochure_system_prompt="""
You are an assistant that analyses the contents of several relevant pages from company website and create a short brochure  about the company for propective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of campany culture,customers and careers jobs if you have the information."""

In [24]:
from scrapper import fetch_website

In [25]:
def stream_brochure(company_name,url,model):
    yield ""
    prompt=f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt+=fetch_website(url)
    if model=="GPT":
        result=stream_gpt(prompt)
    elif model=="Gemini":
        result=stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [26]:
name_input=gr.Textbox(label="Company name")
url_input=gr.Textbox(label="Landing page url")
model_selector=gr.Dropdown(["GPT","Gemini"],label="Select model",value="GPT")
message_output=gr.Markdown(label="Response")
view=gr.Interface(fn=stream_brochure,title="Brochure generator",inputs=[name_input,url_input,model_selector],outputs=[message_output],examples=[["Huggingface","https://huggingface.co","GPT"],["Figma","https://figma.com","Gemini"]],flagging_mode="never")
view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
